# Project · Is the new version better? A/B testing from first principles

From **Neuron Voxel Lab** (https://neuron-voxel-lab.vercel.app). A product team shows 4,000 visitors the old page (A) and 4,000 the new page (B) and counts sign-ups. Is B really better, or did it get lucky? You'll answer that six ways, from the central limit theorem to a permutation test and a Bayesian posterior. Then you'll measure how often such a test can even detect the effect, and watch 'peeking' at results quietly break it. Uses Murphy ch. 2, 4 and 5.

Work through the milestones in order; each builds on the previous cell's variables. Solutions are hidden: double-click a solution cell's title to reveal it.

## Milestone 1 · Simulate the experiment

Simulate 4,000 visitors per arm, where A converts with true probability 10% and B with 11.5% (pretend you don't know these). Report each arm's observed rate and the observed difference.

In [ ]:
# Your turn
import numpy as np
rng = np.random.default_rng(7)
nA = nB = 4000
a = rng.random(nA) < 0.100                     # 1 = signed up
b = rng.random(nB) < 0.115
pA, pB = ...  # TODO: observed conversion rates
diff = pB - pA
print(f"A: {a.sum()} / {nA} = {pA:.4f}    B: {b.sum()} / {nB} = {pB:.4f}    difference {diff:+.4f}")

In [ ]:
# @title Solution 1 (double-click to show the code)
import numpy as np
rng = np.random.default_rng(7)
nA = nB = 4000
a = rng.random(nA) < 0.100                     # 1 = signed up
b = rng.random(nB) < 0.115
pA, pB = a.mean(), b.mean()  # observed conversion rates
diff = pB - pA
print(f"A: {a.sum()} / {nA} = {pA:.4f}    B: {b.sum()} / {nB} = {pB:.4f}    difference {diff:+.4f}")

<details><summary><b>Answer and explanation</b></summary>

Observed: A 395/4000 = 9.88%, B 463/4000 = 11.58%, a difference of +1.70 percentage points. The true difference is +1.5 points, so chance alone moved the estimate by 0.2 points. Every number below is about one question: how much of that +1.70 could be luck?

Expected output:
```
A: 395 / 4000 = 0.0988    B: 463 / 4000 = 0.1158    difference +0.0170
```

</details>

## Milestone 2 · Error bar from the central limit theorem

Each observed rate is an average of 0/1 values, so by the CLT it is roughly Gaussian with variance p(1 − p)/n. Compute the standard error of the difference, a 95% confidence interval, and the z-score.

In [ ]:
# Your turn
se = ...  # TODO: variances of independent averages add
lo, hi = diff - 1.96 * se, diff + 1.96 * se
z = diff / se
print(f"standard error {se:.4f}   95% CI [{lo:+.4f}, {hi:+.4f}]   z = {z:.2f}")

In [ ]:
# @title Solution 2 (double-click to show the code)
se = np.sqrt(pA * (1 - pA) / nA + pB * (1 - pB) / nB)  # variances of independent averages add
lo, hi = diff - 1.96 * se, diff + 1.96 * se
z = diff / se
print(f"standard error {se:.4f}   95% CI [{lo:+.4f}, {hi:+.4f}]   z = {z:.2f}")

<details><summary><b>Answer and explanation</b></summary>

Standard error ≈ 0.0069, so the 95% interval is [+0.0034, +0.0306] and z ≈ 2.46. The logic: an average of n independent 0/1 values has variance p(1 − p)/n (Murphy §2.2), the CLT makes it roughly Gaussian (§2.8.6), and variances of independent things add. The interval excludes 0, so 'no difference' is not a good fit to this data. But it is wide: the true lift could plausibly be anywhere from +0.3 to +3 points.

Expected output:
```
standard error 0.0069   95% CI [+0.0034, +0.0306]   z = 2.46
```

</details>

## Milestone 3 · The same error bar by bootstrap

Don't trust the formula? Resample each arm with replacement 5,000 times, recompute the difference each time, and take the 2.5% and 97.5% percentiles. Compare with step 2.

In [ ]:
# Your turn
boot = np.empty(5000)
for i in range(5000):
    boot[i] = ...  # TODO: resample each arm, recompute
print(f"bootstrap 95% CI [{np.percentile(boot, 2.5):+.4f}, {np.percentile(boot, 97.5):+.4f}]   bootstrap SE {boot.std():.4f}")

In [ ]:
# @title Solution 3 (double-click to show the code)
boot = np.empty(5000)
for i in range(5000):
    boot[i] = rng.choice(b, nB).mean() - rng.choice(a, nA).mean()  # resample each arm, recompute
print(f"bootstrap 95% CI [{np.percentile(boot, 2.5):+.4f}, {np.percentile(boot, 97.5):+.4f}]   bootstrap SE {boot.std():.4f}")

<details><summary><b>Answer and explanation</b></summary>

The bootstrap gives [+0.0035, +0.0307] with SE 0.0069, essentially identical to the formula. The bootstrap pretends your sample is the population and asks how much the statistic wobbles across re-runs (Murphy §4.7.3). It needs no formula, which is its power for statistics that have no neat variance formula, such as medians, ratios or AUC. When a formula exists and its assumptions hold (here, large n), the two agree.

Expected output:
```
bootstrap 95% CI [+0.0035, +0.0307]   bootstrap SE 0.0069
```

</details>

## Milestone 4 · A p-value two ways: permutation and normal

If the page made no difference, the labels A/B are arbitrary. Shuffle the labels 10,000 times and count how often a shuffled difference is at least as extreme as the real one. Compare with the normal-approximation p-value 2·P(Z > |z|).

In [ ]:
# Your turn
from math import erfc, sqrt
pool = np.concatenate([a, b]).astype(float)
perm = np.empty(10000)
for i in range(10000):
    s = rng.permutation(pool)
    perm[i] = ...  # TODO: difference if labels meant nothing
p_perm = np.mean(np.abs(perm) >= abs(diff))
p_norm = erfc(abs(z) / sqrt(2))                      # = 2 * P(Z > |z|)
print(f"permutation p = {p_perm:.4f}    normal-approximation p = {p_norm:.4f}")

In [ ]:
# @title Solution 4 (double-click to show the code)
from math import erfc, sqrt
pool = np.concatenate([a, b]).astype(float)
perm = np.empty(10000)
for i in range(10000):
    s = rng.permutation(pool)
    perm[i] = s[nA:].mean() - s[:nA].mean()  # difference if labels meant nothing
p_perm = np.mean(np.abs(perm) >= abs(diff))
p_norm = erfc(abs(z) / sqrt(2))                      # = 2 * P(Z > |z|)
print(f"permutation p = {p_perm:.4f}    normal-approximation p = {p_norm:.4f}")

<details><summary><b>Answer and explanation</b></summary>

Permutation p ≈ 0.015; normal approximation p ≈ 0.014. The permutation test builds the 'no difference' world directly: if the page didn't matter, which visitor got which page is arbitrary, so shuffling the labels shows the spread of differences that luck alone produces. Only about 1.5% of shuffles are as extreme as +1.70 points. It makes no Gaussian assumption, and with n = 4,000 per arm the CLT version agrees closely.

Expected output:
```
permutation p = 0.0151    normal-approximation p = 0.0140
```

</details>

## Milestone 5 · The Bayesian answer: how likely is B better?

Give each conversion rate a flat Beta(1, 1) prior. The posterior is Beta(1 + successes, 1 + failures). Draw 200,000 samples from each posterior and estimate P(rate_B > rate_A), the expected relative lift, and a 95% credible interval for the difference.

In [ ]:
# Your turn
sA = rng.beta(1 + a.sum(), 1 + nA - a.sum(), 200000)
sB = ...  # TODO: conjugate posterior: Beta(1 + hits, 1 + misses)
d = sB - sA
print(f"P(B better than A) = {np.mean(d > 0):.3f}")
print(f"expected relative lift {np.mean(sB / sA - 1):+.1%}   95% credible interval for the difference [{np.percentile(d, 2.5):+.4f}, {np.percentile(d, 97.5):+.4f}]")

In [ ]:
# @title Solution 5 (double-click to show the code)
sA = rng.beta(1 + a.sum(), 1 + nA - a.sum(), 200000)
sB = rng.beta(1 + b.sum(), 1 + nB - b.sum(), 200000)  # conjugate posterior: Beta(1 + hits, 1 + misses)
d = sB - sA
print(f"P(B better than A) = {np.mean(d > 0):.3f}")
print(f"expected relative lift {np.mean(sB / sA - 1):+.1%}   95% credible interval for the difference [{np.percentile(d, 2.5):+.4f}, {np.percentile(d, 97.5):+.4f}]")

<details><summary><b>Answer and explanation</b></summary>

P(B better) ≈ 0.993, expected relative lift ≈ +17%, 95% credible interval [+0.0034, +0.0306]. With a flat prior and lots of data the credible interval matches the confidence interval numerically, but it means something different: 'given the data, the difference is in here with 95% probability' (Murphy §4.6). Note that 0.993 ≈ 1 − p/2 here. That is a coincidence of flat priors and symmetric large-sample posteriors, not a general identity.

Expected output:
```
P(B better than A) = 0.993
expected relative lift +17.4%   95% credible interval for the difference [+0.0034, +0.0306]
```

</details>

## Milestone 6 · Power: could this test even see the effect?

Suppose the truth really is 10% vs 11.5%. Simulate 2,000 experiments for several sample sizes per arm and record how often the z-test gives p < 0.05. How many visitors per arm do you need for 80% power?

In [ ]:
# Your turn
def power(n, pa=0.100, pb=0.115, sims=2000):
    ka = rng.binomial(n, pa, sims); kb = rng.binomial(n, pb, sims)   # number of sign-ups in each simulated experiment
    ra, rb = ka / n, kb / n
    se = np.sqrt(ra * (1 - ra) / n + rb * (1 - rb) / n)
    return ...  # TODO: fraction of experiments that detect the effect
for n in [1000, 2000, 4000, 6000, 8000, 10000]:
    print(f"n = {n:>5} per arm  ->  power {power(n):.2f}")

In [ ]:
# @title Solution 6 (double-click to show the code)
def power(n, pa=0.100, pb=0.115, sims=2000):
    ka = rng.binomial(n, pa, sims); kb = rng.binomial(n, pb, sims)   # number of sign-ups in each simulated experiment
    ra, rb = ka / n, kb / n
    se = np.sqrt(ra * (1 - ra) / n + rb * (1 - rb) / n)
    return np.mean(np.abs(rb - ra) / se > 1.96)  # fraction of experiments that detect the effect
for n in [1000, 2000, 4000, 6000, 8000, 10000]:
    print(f"n = {n:>5} per arm  ->  power {power(n):.2f}")

<details><summary><b>Answer and explanation</b></summary>

With 4,000 per arm the power is only ≈ 0.57. Even if B truly is 1.5 points better, this experiment would miss it more than 4 times in 10, so this run was fairly lucky to reach significance. You need ≈ 6,500 per arm for 80% power (0.75 at 6,000, 0.87 at 8,000). The standard error shrinks like 1/√n, so halving it costs 4× the visitors. Run this calculation before the experiment, to choose n.

Expected output:
```
n =  1000 per arm  ->  power 0.19
n =  2000 per arm  ->  power 0.34
n =  4000 per arm  ->  power 0.57
n =  6000 per arm  ->  power 0.75
n =  8000 per arm  ->  power 0.87
n = 10000 per arm  ->  power 0.94
```

</details>

## Milestone 7 · Peeking breaks the p-value

Now make A and B identical (both 10%). Run 2,000 experiments of up to 10,000 visitors per arm. Check the p-value after every 500 visitors and stop as soon as p < 0.05. What fraction of experiments falsely declare a winner? Compare with checking once at the end.

In [ ]:
# Your turn
sims, N, step = 2000, 10000, 500
xa = rng.random((sims, N)) < 0.10; xb = rng.random((sims, N)) < 0.10   # no real difference
ever, final = np.zeros(sims, bool), None
for n in range(step, N + 1, step):
    ra, rb = xa[:, :n].mean(1), xb[:, :n].mean(1)
    se = np.sqrt(ra * (1 - ra) / n + rb * (1 - rb) / n)
    sig = np.abs(rb - ra) / se > 1.96
    ever | = ...  # TODO: stop at the first "significant" look
    final = sig
print(f"false positives, one look at the end: {final.mean():.3f}")
print(f"false positives, peeking every {step}:  {ever.mean():.3f}")

In [ ]:
# @title Solution 7 (double-click to show the code)
sims, N, step = 2000, 10000, 500
xa = rng.random((sims, N)) < 0.10; xb = rng.random((sims, N)) < 0.10   # no real difference
ever, final = np.zeros(sims, bool), None
for n in range(step, N + 1, step):
    ra, rb = xa[:, :n].mean(1), xb[:, :n].mean(1)
    se = np.sqrt(ra * (1 - ra) / n + rb * (1 - rb) / n)
    sig = np.abs(rb - ra) / se > 1.96
    ever |= sig  # stop at the first "significant" look
    final = sig
print(f"false positives, one look at the end: {final.mean():.3f}")
print(f"false positives, peeking every {step}:  {ever.mean():.3f}")

<details><summary><b>Answer and explanation</b></summary>

Checking once at the end gives a false-positive rate of exactly the promised 5%. Peeking every 500 visitors and stopping at the first p < 0.05 gives ≈ 24%: almost 1 in 4 'winners' are pure noise. Each look is another chance for the random walk of the difference to wander past the line, and once you stop you never see it wander back. Fixes: fix n in advance (step 6), use sequential tests designed for repeated looks, or go Bayesian and report the posterior, which doesn't depend on your stopping plan.

Expected output:
```
false positives, one look at the end: 0.050
false positives, peeking every 500:  0.237
```

</details>

## Milestone 8 · What the numbers do and don't say

Your manager reads step 4 and says: 'p = 0.03, so there's a 97% chance B is better.' Correct the sentence, then say which of your results actually answers the manager's question.

✍️ *Write your answer here.*

<details><summary><b>Model answer</b></summary>

A p-value is P(data at least this extreme | no difference). It is not P(no difference | data), and 1 − p is not the chance that B is better: that's the prosecutor's fallacy (Murphy §5.5.4). The quantity the manager wants, P(B better | data), is the Bayesian posterior probability from step 5. With a flat prior and this much data it happens to be close to 1 − p/2, but in general it depends on the prior, and the two numbers answer different questions. Also say how big the effect is (the interval from steps 2–3), not just whether it's 'significant'. And only quote a p-value if you decided the sample size in advance: step 7 shows that peeking inflates false positives far above 5%.

</details>